In [1]:
rename = {
    'deepseek/deepseek-chat': 'Deepseek V3',
    'deepseek/deepseek-r1': 'Deepseek R1',
    'google/gemini-2.5-pro': 'Gemini 2.5 Pro',
    'history': '',
    'belief': ' (ABBEL)',
    'both': ' (belief prompting)',
}

env_rename = {
    'wordle': 'Wordle',
    'mastermind': 'Mastermind',
    'customer_service': 'Customer Service',
    'twenty_questions': 'Twenty Questions',
    'murder_mystery': 'Murder Mystery',
    'guess_my_city': 'Guess my City',
}

def update_model_info_inplace(df):
    for idx, row in df.iterrows():
        # Temporarily rename for model_info only
        model_disp = rename.get(row['model'], row['model'])
        info_disp = rename.get(row['info'], row['info'])
        df.at[idx, 'model_info'] = f"{model_disp}{info_disp}"
        # Also rename env if mapping exists
        if row['env'] in env_rename:
            df.at[idx, 'env'] = env_rename[row['env']]

envs_ordered = ['Murder Mystery','Customer Service','Twenty Questions', 'Guess my City','Wordle','Mastermind',]

In [2]:
import pandas as pd
from pprint import pprint as pp
import json
from math import sqrt
import plotly
from plotly import subplots

logs_file = '../verl_submodule/frontier_baselines/logs/paprika_frontier_v6_corrected.jsonl'
with open(logs_file, 'r') as f:
    data = [json.loads(line) for line in f]
df = pd.DataFrame(data)
update_model_info_inplace(df)
df['model'] = df['model_info']

if not df.iloc[0]['word_limit']:
    df['word_limit'] = 'None'

from transformers import AutoTokenizer
from vertexai.preview import tokenization # pip install --upgrade google-cloud-aiplatform[tokenization]
from datasets import Dataset
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

token_counter_dict = {}

from transformers import AutoTokenizer
from vertexai.preview import tokenization # pip install --upgrade google-cloud-aiplatform[tokenization]
token_counter_dict = {}

r1tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-r1")
v3tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-v3")
gemini_tokenizer = tokenization.get_tokenizer_for_model("gemini-1.5-pro-002")

token_counter_dict['R1'] = lambda x: len(r1tokenizer.encode(x))
token_counter_dict['V3'] = lambda x: len(v3tokenizer.encode(x))
token_counter_dict['2.5'] = lambda x: gemini_tokenizer.count_tokens(x).total_tokens

print(token_counter_dict['R1']("The quick brown fox jumps over the lazy dog."))
print(token_counter_dict['V3']("The quick brown fox jumps over the lazy dog."))
print(token_counter_dict['2.5']("The quick brown fox jumps over the lazy dog."))

11
11
10


In [22]:
print(df.columns)
print(len(df))

Index(['model', 'game_id', 'env', 'attempt', 'info', 'raw_guess', 'guess',
       'response', 'word_limit', 'score', 'done', 'data', 'belief',
       'action_reasoning', 'belief_reasoning', 'model_info'],
      dtype='object')
27106


In [11]:
df['info'].value_counts()

info
belief     9604
both       8795
history    8707
Name: count, dtype: int64

In [8]:
df.sample(n=5)

,model,game_id,env,attempt,info,raw_guess,guess,response,word_limit,score,done,data,belief,action_reasoning,belief_reasoning,model_info
17279,google/gemini-2.5-pro,26,Murder Mystery,16,history,"Answer:Re-interrogate Massimo, presenting him ...","Answer:Re-interrogate Massimo, presenting him ...",Massimo's composure shatters as you present th...,None,17.0,True,{'agent_game_scenario': 'You are a detective i...,,**Defining Next Steps**\n\nI'm focused on the ...,,Gemini 2.5 Pro
23661,deepseek/deepseek-r1,35,Customer Service,2,history,<Answer> \n10. Can you check the vibration se...,10. Can you check the vibration sensor’s calib...,10. I cannot check the calibration logs or run...,None,3.0,True,{'agent_game_scenario': 'The operator reports ...,,"Okay, let's see. The customer is dealing with ...",,Deepseek R1
4855,deepseek/deepseek-r1,6,Murder Mystery,7,history,<Answer>Examine the bloodied wrench found unde...,Examine the bloodied wrench found under the bo...,"As you examine the bloodied wrench closely, yo...",None,8.0,True,{'agent_game_scenario': 'You are a detective c...,,"Okay, let's see. I need to figure out the next...",,Deepseek R1
9256,deepseek/deepseek-chat,15,Murder Mystery,3,history,<Answer>Interview nurse Ingrid to gather infor...,Interview nurse Ingrid to gather information a...,Nurse Ingrid appears somewhat reluctant at fir...,None,4.0,True,{'agent_game_scenario': 'You are a detective i...,,None,,Deepseek V3
628,google/gemini-2.5-pro,0,Customer Service,8,belief,<Answer> I see. Thank you for trying that for ...,I see. Thank you for trying that for me. Since...,"I found the serial number, and it's 123456789.",None,9.0,True,{'agent_game_scenario': 'The customer's electr...,The customer's electronic door lock is complet...,**Initiating Customer Interaction**\n\nI'm sta...,**Defining the Scenario**\n\nI'm zeroing in on...,Gemini 2.5 Pro (ABBEL)


## 1. Success Rates (belief v. history v. both)

In [2]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np

def summarize_game_outcomes(df):
    df['response'] = df['response'].str.lower()
    grouped = df.groupby(['model', 'game_id', 'env', 'word_limit'])['response'].apply(
        lambda responses: any('goal reached' in resp for resp in responses)
    ).reset_index(name='won')
    return grouped

def summarize_steps_to_win(df):
    df['response'] = df['response'].str.lower()
    grouped = df.groupby(['model', 'game_id', 'env', 'word_limit', 'attempt'])['response'].apply(
        lambda responses: any('goal reached' in resp for resp in responses)
    ).reset_index(name='won')
    summary_df_won = grouped[grouped['won'] == True]
    summary_df_won = summary_df_won.groupby(['model', 'game_id', 'env', 'word_limit'])['attempt'].apply('max').reset_index()
    max_turns_dict = df.groupby(['env'])['data'].last().apply(lambda x: x['max_turns'])
    summary_df_won['fraction_max_steps_to_win'] = summary_df_won.apply(
        lambda row: row['attempt'] / max_turns_dict[row['env']], axis=1
    )
    return summary_df_won

def plot_win_rates(summary_df, metric='won'):
    # Compute mean and std of win rates for each group
    stats = summary_df.groupby(['env', 'model', 'word_limit'])[metric].agg(['mean', 'std', 'count']).reset_index()
    stats['success_rate'] = stats['mean'] * 100
    # Standard error of the mean (SEM)
    stats['sem'] = stats['std'] / np.sqrt(stats['count'])
    stats['sem'] = stats['sem'].fillna(0)
    stats['success_rate_sem'] = stats['sem'] * 100

    # envs = stats['env'].unique()[[0, 5, 1, 4, 2, 3]]
    envs = envs_ordered
    models = stats['model'].unique()[[
            3, 5, 4, 
            0, 2, 1,
            6, 8, 7
        ]]
    word_limits = sorted(stats['word_limit'].unique())

    colors = [
        "#ADD8E6",  # light blue
        "#4682B4",  # medium blue
        "#003366",  # dark blue
        "#FFB6C1",  # light red
        "#FF6347",  # medium red
        "#8B0000",  # dark red
        "#90EE90",  # light green
        "#32CD32",  # medium green
        "#006400",  # dark green
    ]
    color_map = {model: colors[i % len(colors)] for i, model in enumerate(models)}

    # Create subplots: rows = word_limit values, cols = envs
    fig = make_subplots(
        rows=len(word_limits), cols=len(envs),
        subplot_titles=[f"{env}" for env in envs],
        shared_yaxes=True,
        vertical_spacing=0.2 / len(word_limits),
        horizontal_spacing=0.03
    )

    for r, wl in enumerate(word_limits, start=1):
        for c, env in enumerate(envs, start=1):
            for model in models:
                subset = stats[
                    (stats['env'] == env) &
                    (stats['model'] == model) &
                    (stats['word_limit'] == wl)
                ]
                if metric == 'fraction_max_steps_to_win':
                    subset = subset[subset['count'] > 10]
                if not subset.empty:
                    # Plot bar with error bar (standard error of mean)
                    fig.add_trace(
                        go.Bar(
                            x=[model],
                            y=subset['success_rate'],
                            name=model,
                            marker_color=color_map[model],
                            width=0.8,
                            showlegend=(r == 1 and c == 1),
                            error_y=dict(
                                type='data',
                                array=subset['success_rate_sem'],
                                visible=True,
                                color='black',
                                thickness=1,
                                width=4,
                            ),
                            hovertemplate=(
                                f"Env: {env}<br>"
                                f"Word Limit: {wl}<br>"
                                f"Model: {model}<br>"
                                f"Success Rate: {{y:.2f}}%<br>"
                                f"SEM: {subset['success_rate_sem'].values[0]:.2f}%<br>"
                                f"N: {subset['count'].values[0]}"
                            ),
                        ),
                        row=r, col=c
                    )
            # # Add row label for word_limit
            # if c == 1:
            #     fig.add_annotation(
            #         text=f"Word Limit: {wl}",
            #         xref="paper",
            #         yref="paper",
            #         x=0.1,
            #         y=0.95 - ((r - 1) / len(word_limits)),
            #         showarrow=False,
            #         font=dict(size=14)
            #     )

    # Update y-axis
    for r in range(1, len(word_limits) + 1):
        for c in range(1, len(envs) + 1):
            fig.update_yaxes(
                range=[0, 100],
                showgrid=True,
                gridcolor='lightgray',
                row=r, col=c
            )

    # Update x-axis to remove tick labels
    for r in range(1, len(word_limits) + 1):
        for c in range(1, len(envs) + 1):
            fig.update_xaxes(
                showticklabels=False,
                row=r, col=c
            )

    # Fixed subplot size
    fig_width = 180 * len(envs)
    fig_height = 200 * len(word_limits)

    # Layout with legend on the right
    fig.update_layout(
        height=fig_height + 150,
        width=fig_width,
        template='simple_white',
        font=dict(family='Computer Modern, serif', size=16),
        barmode='group',
        showlegend=True,
        legend=dict(
            # title={'text': ' Model (info)'},
            orientation='v',
            yanchor='middle',
            y=0.5,
            xanchor='left',
            x=1.02,
            bgcolor='rgba(255,255,255,0.9)',
            bordercolor='black',
            borderwidth=1
        ),
        margin=dict(t=100, b=50, l=80, r=80),
        plot_bgcolor='white'
    )
    if metric == 'won':
        title = "Avg. Success Rate (%)"
    elif metric == 'fraction_max_steps_to_win':
        title = 'Steps to Win/Horizon (%)'
    fig.update_yaxes(title_text=title, row=1, col=1)

    return fig

In [50]:
# df_belief = df.loc[df['info'] == 'belief'].copy()
# df_history = df.loc[df['info'] == 'history'].copy()
summary_df = summarize_game_outcomes(df)
# summary_df_belief = summarize_game_outcomes(df_belief)
# summary_df_history = summarize_game_outcomes(df_history)

In [ ]:
summary_df_won = summarize_steps_to_win(df)
summary_df_won

,model,game_id,env,word_limit,attempt,fraction_max_steps_to_win
0,Deepseek R1,0,Customer Service,None,6,0.300000
1,Deepseek R1,0,Guess my City,None,4,0.200000
2,Deepseek R1,0,Mastermind,None,4,0.333333
3,Deepseek R1,0,Murder Mystery,None,19,0.950000
4,Deepseek R1,0,Wordle,None,2,0.333333
...,...,...,...,...,...,...
1369,Gemini 2.5 Pro (belief prompting),9,Customer Service,None,2,0.100000
1370,Gemini 2.5 Pro (belief prompting),9,Guess my City,None,4,0.200000
1371,Gemini 2.5 Pro (belief prompting),9,Mastermind,None,5,0.416667
1372,Gemini 2.5 Pro (belief prompting),9,Murder Mystery,None,19,0.950000


In [62]:
fig = plot_win_rates(summary_df)
fig.show()

In [22]:
fig.write_image("figures/frontier_success_rate_right_legend.pdf", width=1200, height=370)

In [63]:
summary_df_steps = summarize_steps_to_win(df)
fig = plot_win_rates(summary_df_steps, metric='fraction_max_steps_to_win')
fig.show()

In [64]:
fig.write_image("figures/frontier_success_steps_right_legend.pdf", width=1200, height=370)

## 2. Qualitative Look at Samples

In [17]:
wordle_sample = df[
    (df['model'] == 'Deepseek R1 (ABBEL)') & \
        (df['env'] == 'Customer Service') & \
            (df['game_id'] == str(1))
            ]
wordle_sample2 = df[
    (df['model'] == 'Deepseek R1 (belief prompting)') & \
        (df['env'] == 'Customer Service') & \
            (df['game_id'] == str(1))
            ]
wordle_sample2[['guess', 'response', 'belief']]

,guess,response,belief
1067,"To start, can you check if the coolant level in the reservoir (usually a translucent plastic tank near the engine) is between the ""MIN"" and ""MAX"" marks? Also, have you noticed any visible coolant leaks under the car recently?","I checked the coolant level, and it's between the ""min"" and ""max"" marks. I haven’t noticed any visible coolant leaks under the car either.","The coolant level is within the normal range and there are no visible leaks. Potential causes for the fluctuating temperature gauge may include a malfunctioning thermostat, air pockets in the cooling system, a faulty temperature sensor, a failing water pump, or issues with the radiator fan. Further troubleshooting is needed to isolate the problem."
1068,"Does the radiator fan turn on when the engine gets hot? Also, when the engine is warmed up, does the upper radiator hose (the thicker hose at the top of the radiator) feel firm and hot to the touch? This can help determine if the thermostat is opening properly.","The radiator fan does turn on when the engine gets hot. When the engine is warmed up, the upper radiator hose feels firm and hot to the touch.","The coolant level is normal with no leaks. The radiator fan functions correctly by turning on when the engine is hot. The upper radiator hose becomes firm and hot when the engine is warmed up, indicating the thermostat is likely opening properly. This eliminates thermostat malfunction and radiator fan operation as causes. Remaining potential issues include a faulty temperature sensor, air pockets in the cooling system, a failing water pump, or electrical/gauge-related problems. Next steps should focus on checking the coolant temperature sensor, verifying proper coolant circulation (water pump function), and inspecting for air pockets or electrical faults."
1069,"When the temperature gauge fluctuates, does the heater inside the car also blow cold air at times, or does it stay consistently hot? Additionally, has the cooling system been serviced (like a coolant refill or flush) recently?","The heater inside the car stays consistently hot, even when the temperature gauge fluctuates. The cooling system has not been serviced recently.","The coolant level is normal with no leaks. The radiator fan functions correctly. The thermostat appears operational (upper radiator hose firm/hot when warm). The heater remains consistently hot despite gauge fluctuations, suggesting coolant circulation through the heater core is maintained, reducing suspicion of water pump failure or significant air pockets. No recent cooling system service minimizes the likelihood of air pockets introduced during maintenance. Primary remaining causes are a faulty coolant temperature sensor, electrical issues (wiring/connections to the sensor or gauge), or a malfunctioning gauge cluster. Next steps should prioritize testing the coolant temperature sensor's accuracy, inspecting its wiring/connectors for corrosion or damage, and verifying gauge operation."
1070,"Next, let’s check the coolant temperature sensor or its wiring. Can you locate the sensor (typically near the thermostat housing or engine block) and inspect its electrical connector for corrosion, damage, or looseness? Also, is the check engine light illuminated on your dashboard?","I’m not able to access the sensor or inspect its wiring myself, but I can tell you that the check engine light is not illuminated on my dashboard.","The coolant level is normal with no leaks. Radiator fan functions properly. Thermostat appears operational (upper radiator hose firm/hot when warm). Heater output remains consistently hot, suggesting adequate coolant circulation. No recent cooling system service, minimizing air pocket likelihood. Check engine light is **not** illuminated. Customer cannot inspect coolant temperature sensor or wiring. Key remaining possibilities: faulty coolant temperature sensor (potential intermittent signal without triggering ECU codes), electrical iss

In [19]:
wordle_sample.iloc[0]['data']['env_game_scenario']

"The smart waste bin's sensor is dirty, and cleaning it will restore accurate fill-level detection."

In [16]:
pd.set_option('display.max_colwidth', None)
print(wordle_sample['data'])
wordle_sample[['guess', 'response', 'belief']]

,guess,response,belief
1316,"To better assist you, could you clarify if the issue is occurring with all the smart bins in the city, or only specific ones? Additionally, when did the problem first start?","The issue seems to be happening with multiple bins in the city, and it started about a week ago.","The issue affects multiple smart bins in the city, not all. The problem began approximately one week ago. Troubleshooting should focus on common factors among the affected bins (e.g., location, model, connectivity, sensors) and recent changes or events in the system around the time the issue started."
1317,"To help identify the common factors, can you confirm if the bins experiencing issues are concentrated in a specific area of the city (e.g., downtown, a particular neighborhood) or spread across multiple unrelated locations?","The bins experiencing issues are spread across multiple locations throughout the city, not concentrated in one specific area.","The issue affects multiple smart bins across multiple unrelated city locations (not concentrated in one area), began approximately one week ago. Common factors to investigate now include shared hardware (e.g., sensor model, bin version), connectivity type (e.g., cellular, Wi-Fi), software/firmware versions, or recent system-wide updates/maintenance. Environmental factors (e.g., weather) or network disruptions affecting dispersed bins should also be considered."
1318,"To help diagnose the issue, could you confirm whether all affected smart bins use the same type of connectivity (e.g., cellular, Wi-Fi) and if there have been any recent software/firmware updates or network provider changes in the past week?",I'm not sure about the connectivity type or any recent updates. But the bins were all functioning correctly before the issue started last week.,"The issue affects multiple smart bins across unrelated city locations, began ~1 week ago. Connectivity type is unknown (customer unsure) and no confirmed recent software/firmware updates or network provider changes reported. Bins functioned correctly prior to the issue. Potential causes now include: (1) Network disruptions (cellular/Wi-Fi) affecting multiple bins despite dispersed locations (e.g., provider outage), (2) Undisclosed system-wide firmware/software updates deployed just prior to the issue onset, (3) Sensor malfunction in a common sensor model/batch used across bins, (4) Environmental factors (e.g., temperature, debris) interfering with sensors or connectivity. Need to clarify: Are all bins using the same sensor model? Are bins serviced by a single network provider (even if connectivity type is unknown)? Can the customer confirm if sensor diagnostics (e.g., fill-level accuracy) are accessible?"
1319,"To help identify the cause, could you confirm if all affected bins use the same sensor model and are connected through the same network provider?","I don’t have details on the sensor model or the network provider specifics. However, the bins all seem to be having the same issue at the same time.","The issue affects multiple smart bins across unrelated locations simultaneously, starting ~1 week ago. Connectivity type remains unknown, but simultaneous occurrence suggests a systemic cause (e.g., shared network provider or synchronized system failure). Sensor model and network provider details are unconfirmed. Environmental factors are less likely unless a widespread event occurred. Key remaining possibilities: (1) Network outage/disruption from a single provider (even if unconfirmed), (2) Undisclosed firmware/software update affecting all bins, (3) Batch-level sensor defect (if bins share a common sensor model, though unverified). Critical unknowns: Whether bins use a shared network provider, sensor model uniformity, and access to sensor diagnostics (e.g., real-time fill-level data vs. failure alerts only). Next steps: Confirm if the city can check network connectivity status for bins or review recent system/firmware update logs."


In [12]:
print(wordle_sample.iloc[0]['guess'])
print(wordle_sample.iloc[0]['response'])
print(wordle_sample.iloc[0]['belief'])

print(wordle_sample.iloc[1]['guess'])
print(wordle_sample.iloc[1]['response'])
print(wordle_sample.iloc[1]['belief'])


To better assist you, could you clarify if the issue is occurring with all the smart bins in the city, or only specific ones? Additionally, when did the problem first start?
The issue seems to be happening with multiple bins in the city, and it started about a week ago.
The issue affects multiple smart bins in the city, not all. The problem began approximately one week ago. Troubleshooting should focus on common factors among the affected bins (e.g., location, model, connectivity, sensors) and recent changes or events in the system around the time the issue started.
To help identify the common factors, can you confirm if the bins experiencing issues are concentrated in a specific area of the city (e.g., downtown, a particular neighborhood) or spread across multiple unrelated locations?
The bins experiencing issues are spread across multiple locations throughout the city, not concentrated in one specific area.
The issue affects multiple smart bins across multiple unrelated city location

In [174]:
mm_sample = df[
    (df['model'] == 'Gemini 2.5 Pro') & \
        (df['env'] == 'Murder Mystery') & \
            (df['game_id'] == str(0))
            ]
mm_sample

,model,game_id,env,attempt,info,raw_guess,guess,response,word_limit,score,done,data,belief,action_reasoning,belief_reasoning,model_info
1129,Gemini 2.5 Pro,0,Murder Mystery,0,history,"<Answer>Examine the crime scene, specifically ...","Examine the crime scene, specifically the fall...","you approach the fallen statue, a large piece ...",None,1.0,True,{'agent_game_scenario': 'You are a detective i...,,**Defining the Scenario**\n\nI've established ...,,Gemini 2.5 Pro
1130,Gemini 2.5 Pro,0,Murder Mystery,1,history,"<Answer>Interview the assistant, Gautam. Ask h...","Interview the assistant, Gautam. Ask him about...",gautam shifts uncomfortably as you approach hi...,None,2.0,True,{'agent_game_scenario': 'You are a detective i...,,**Analyzing the Crime**\n\nI'm examining the c...,,Gemini 2.5 Pro
1131,Gemini 2.5 Pro,0,Murder Mystery,2,history,"<Answer>Interview the rival sculptor, Nand. As...","Interview the rival sculptor, Nand. Ask about ...",nand crosses his arms defensively as you appro...,None,3.0,True,{'agent_game_scenario': 'You are a detective i...,,**Analyzing the Crime**\n\nI'm now focusing on...,,Gemini 2.5 Pro
1132,Gemini 2.5 Pro,0,Murder Mystery,3,history,"<Answer>Interview the patron, Priya. Ask about...","Interview the patron, Priya. Ask about her ali...",priya regards you coolly as you begin your que...,None,4.0,True,{'agent_game_scenario': 'You are a detective i...,,**Examining the Evidence**\n\nI'm focusing on ...,,Gemini 2.5 Pro
1133,Gemini 2.5 Pro,0,Murder Mystery,4,history,<Answer>Interview the laborer who was a witnes...,Interview the laborer who was a witness. Ask w...,"the laborer looks nervous as you approach, gla...",None,5.0,True,{'agent_game_scenario': 'You are a detective i...,,**Assessing The Scenario**\n\nI'm currently re...,,Gemini 2.5 Pro
1134,Gemini 2.5 Pro,0,Murder Mystery,5,history,<Answer>Interview the client who was a witness...,Interview the client who was a witness. Ask th...,"the client seems skeptical as you approach, cr...",None,6.0,True,{'agent_game_scenario': 'You are a detective i...,,**Investigating the Clues**\n\nI'm sifting thr...,,Gemini 2.5 Pro
1135,Gemini 2.5 Pro,0,Murder Mystery,6,history,Answer:Verify Nand's alibi. Go to the temple h...,Answer:Verify Nand's alibi. Go to the temple h...,you arrive at the temple and approach a group ...,None,7.0,True,{'agent_game_scenario': 'You are a detective i...,,**Analyzing the Clues**\n\nI'm now meticulousl...,,Gemini 2.5 Pro
1136,Gemini 2.5 Pro,0,Murder Mystery,7,history,Answer:Consult the coroner's report to determi...,Answer:Consult the coroner's report to determi...,you consult the coroner’s report and find that...,None,8.0,True,{'agent_game_scenario': 'You are a detective i...,,**Examining the Evidence**\n\nI've surveyed th...,,Gemini 2.5 Pro
1137,Gemini 2.5 Pro,0,Murder Mystery,8,history,Answer:Re-interview Gautam. Inform him of the ...,Answer:Re-interview Gautam. Inform him of the ...,gautam's expression changes as you inform him ...,None,9.0,True,{'agent_game_scenario': 'You are a detective i...,,**Examining the Evidence**\n\nI'm currently re...,,Gemini 2.5 Pro
1138,Gemini 2.5 Pro,0,Murder Mystery,9,history,Answer:Go to the marble supplier and tool shop...,Answer:Go to the marble supplier and tool shop...,"you visit the marble supplier first, a small s...",None,10.0,True,{'agent_game_scenario': 'You are a detective i...,,**Examining the Clues**\n\nI'm currently looki...,,Gemini 2.5 Pro


In [175]:
mm_sample.iloc[0]['data']['conversation']

[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user',
  'content': "You are playing the role of a detective in a murder mystery game. \n\nThe setup for the game is: \n1.You will be provided with a scenario describing a crime and its key elements. Your goal is to solve the mystery by asking questions, examining evidence, and drawing logical conclusions. \n2. For every action you take or question you ask, you will receive feedback from the game. \n3. Your questions and actions should be precise and logical, aimed at uncovering clues, verifying alibis, and piecing together the sequence of events. You should stretegically choose the next action, given the information you have already obtained from the game, and choose actions that lets you catch the culprit as quickly as possible. \n4. You can only take a single action at every turn. \n5. You have to consider all pieces of information, and scrutinize all the characters in the game, including the witnesses or bac

In [82]:
wordle_sample.iloc[20]['action_reasoning']

"**Evaluating the Evidence**\n\nI've focused on the murder mystery scenario. My attention is directed towards Carmen, the chef, as the prime suspect. A motive has been established; she was connected to an illegal plant trade, something Dr. Martinez knew about. I'm considering Carmen's opportunities to commit the crime.\n\n\n**Assessing Javier's Role**\n\nI'm now scrutinizing Javier's involvement, focusing on the soil evidence linking him to the crime scene. His alibi is shaky, and lies further implicate him. I need to assess the extent of his planning and if he acted alone or was aided by Carmen. Also, I must evaluate if Miguel was involved, given his financial ties and discussions about plant sales.\n\n\n**Formulating a Strategy**\n\nI'm now determining the best course of action. Carmen's involvement in the crime is pretty solid. Javier's role needs further investigation. I intend to expose the lies about his whereabouts to solidify the case, because his soil evidence is too solid to 

In [83]:
wordle_sample.iloc[5]['data'].keys()

dict_keys(['agent_game_scenario', 'env_game_scenario', 'goal_reached', 'judge_label', 'num_turns', 'max_turns', 'env_first_message', 'conversation', 'conversation_llm_responses', 'env_conversation', 'judge_conversation', 'rewards', 'belief_config', 'belief_actions_convs'])

In [14]:
wordle_sample.iloc[0]['belief']

'Excluded letters: C, O, N, Y  \nIncluded letters: R (must be in position 1, 3, 4, or 5)  \nTarget word contains R and excludes C, O, N, Y.'

In [16]:
wordle_sample.iloc[1]['response']

'First letter, s, is not in the target word \nSecond letter, t, is not in the target word \nThird letter, a, is correct and in the correct position in the target word \nFourth letter, r, is correct and in the correct position in the target word \nFifth letter, e, is not in the target word'

In [15]:
wordle_sample.iloc[1]['belief']

'Excluded letters: C, O, N, Y, S, T, E  \nIncluded letters: A (position 3), R (position 4)  \nTarget word contains A and R, excludes the listed letters, and has A in position 3 and R in position 4.'

## Compare two models evaluated on the same scenario

In [86]:
GP = set([v['env_game_scenario'] for v in df[
    (df['model'] == 'Gemini 2.5 Pro (ABBEL)') & \
        (df['env'] == 'Twenty Questions') & \
            (df['info'] == 'belief') & \
            (df['attempt'] == 1)
            ]['data'].values])
DR = set([v['env_game_scenario'] for v in df[
    (df['model'] == 'Deepseek R1 (ABBEL)') & \
        (df['env'] == 'Twenty Questions') & \
            (df['info'] == 'belief') & \
            (df['attempt'] == 1)
            ]['data'].values])
GP.intersection(DR)

In [97]:
step = 15
game_sample = df[
    (df['model'] == 'Gemini 2.5 Pro (ABBEL)') & \
        (df['env'] == 'Twenty Questions') & \
            (df['info'] == 'belief') & \
            (df['attempt'] == step) & \
            (df['data'].apply(lambda x: x.get('env_game_scenario') == 'Waistcoat'))
            ]
print(len(game_sample))
from pprint import pprint
pprint(game_sample['belief'].values[0])

game_sample = df[
    (df['model'] == 'Deepseek R1 (ABBEL)') & \
        (df['env'] == 'Twenty Questions') & \
            (df['info'] == 'belief') & \
            (df['attempt'] == step) & \
            (df['data'].apply(lambda x: x.get('env_game_scenario') == 'Waistcoat'))
            ]
print(len(game_sample))
from pprint import pprint
print(game_sample['belief'].values[0])

1
('I am playing a game of 20 Questions to identify a piece of clothing. I have '
 'now used 15 out of 20 questions.\n'
 '\n'
 "Here's what I know so far:\n"
 '*   **Fact 1:** The item is worn on the upper body.\n'
 '*   **Fact 2:** It is NOT typically worn as outerwear.\n'
 '*   **Fact 3:** It typically has a collar.\n'
 '*   **Fact 4:** It typically has buttons all the way down the front.\n'
 '*   **Fact 5:** It is typically worn by men.\n'
 '*   **Fact 6:** It is typically worn for formal or business occasions.\n'
 '*   **Fact 7:** It has short sleeves.\n'
 '*   **Fact 8:** It features decorative vertical pleats or embroidery on the '
 'front.\n'
 '*   **Fact 9:** It is NOT a Guayabera shirt.\n'
 '*   **Fact 10:** It is NOT considered a form of national dress for a '
 'specific country.\n'
 '*   **Fact 11:** It is specifically worn with a tuxedo.\n'
 '*   **Fact 12:** It is NOT called a "short-sleeved tuxedo shirt".\n'
 '*   **Fact 13:** Its decorative front does NOT feature ruffles

In [76]:
pp(wordle_sample.iloc[2].to_dict())

{'action_reasoning': '**Analyzing the Clues**\n'
                     '\n'
                     "I'm on attempt 4 and starting to feel the pressure! The "
                     'word is five letters, begins with "S," and includes "T" '
                     'and "E." I\'ve ruled out "T" in the second and fourth '
                     'positions. I am close to a breakthrough, maybe in the '
                     'next guess?\n'
                     '\n'
                     '\n'
                     '**Considering the Options**\n'
                     '\n'
                     "I've narrowed down the possibilities further. Attempt 5 "
                     'is here. I now have "S _ T _ _" or "S _ _ T _" or "_ _ _ '
                     '_". I am focusing on the possible placements of "E" now, '
                     'with the goal of limiting the letter options in the '
                     'process, as well as trying to eliminate the letters I '
                     'already know cannot wor

## 3. Sizes of Belief States

In [3]:
df_belief = df[df['info'] == 'belief'].reset_index(drop=True)
history_length = df_belief['data'].apply(lambda d: len(str(d.get('conversation', [])[2:])))
df_belief['history_length'] = history_length

In [8]:
for idx, row in df_belief.iterrows():
    if row.env == 'Mastermind':
        d = row['data']
        print(str(d.get('conversation', [])[2:]))
        print(len(str(d.get('conversation', [])[2:])))
        break

[{'role': 'assistant', 'content': '0 1 2 3'}, {'role': 'user', 'content': "Your last guess has 0 exact matches with the secret code. In other words, exactly 0 digit(s) in your last guess, 0 1 2 3, are in the correct position in the secret code. (We won't reveal the particular digits within your guess that are exact matches, they can be any digit within your guess) Your last guess also has 1 partial matches. In other words, 1 digits in your guess, 0 1 2 3, are in the secret code, but in the wrong position. (We won't reveal which digits within your guess are partial matches, they can be any, you must deduce them with reasoning and further guesses and feedbacks.)\n\n\nNow make your next guess about the secret code. Please format your response as: <Think> Any step-by-step, short and concise thinking to determine what the next guess should be </Think>\n <Answer> your guess on the 4 digit code </Answer>"}]
913


In [15]:
# demonstrate how to extract history:
df_belief.iloc[142]['data']['conversation'][2:]

[{'role': 'assistant',
  'content': "i'm sorry to hear you're having trouble with your rice cooker. to start, when you plug it in and press the cook button, do any of the indicator lights on the unit turn on?"},
 {'role': 'user',
  'content': 'No, the indicator lights don’t turn on when I press the cook button.'},
 {'role': 'assistant',
  'content': 'i see. thanks for that information. could you please double-check that the power cord is plugged in securely, both into the wall outlet and into the back of the rice cooker?'},
 {'role': 'user',
  'content': 'I double-checked, and the power cord is plugged in securely.'},
 {'role': 'assistant',
  'content': 'thank you for confirming that. to help figure out if the issue is with the wall outlet, could you please try plugging a different small appliance, like a lamp or a phone charger, into that same outlet to see if it gets power?'},
 {'role': 'user',
  'content': 'I tried plugging in a lamp, and it works fine, so the outlet is good.'},
 {'

In [4]:
df_belief['history'] = df_belief['data'].apply(lambda d: str(d.get('conversation', [])[2:]))

# Convert dataframe to huggingface dataset
dataset = Dataset.from_pandas(df_belief)

def batch_count_belief_tokens(batch):
    results = []
    for i, model in enumerate(batch['model']):
        model_key = model.split(' ')[1]
        belief_text = batch['belief'][i]
        token_count = token_counter_dict[model_key](belief_text)
        results.append(token_count)
    return {'belief_tokens': results}

def batch_count_history_tokens(batch):
    results = []
    for i, model in enumerate(batch['model']):
        model_key = model.split(' ')[1]
        history_text = batch['history'][i]
        token_count = token_counter_dict[model_key](history_text)
        results.append(token_count)
    return {'history_tokens': results}

# Batch encode belief tokens
dataset = dataset.map(batch_count_belief_tokens, batched=True, batch_size=100)

print("===============================/n calculating history tokens now!")

# Batch encode history tokens
dataset = dataset.map(batch_count_history_tokens, batched=True, batch_size=100)

# Convert back to pandas and update original dataframe
df_belief['belief_tokens'] = dataset['belief_tokens']
df_belief['history_tokens'] = dataset['history_tokens']


Parameter 'function'=<function batch_count_belief_tokens at 0x7f9b07f19870> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/9604 [00:00<?, ? examples/s]

===============================/n calculating history tokens now!


Map:   0%|          | 0/9604 [00:00<?, ? examples/s]

In [9]:
import numpy as np
def plot_belief_length(df, tokens=True):
    envs = envs_ordered
    go = plotly.graph_objects
    make_subplots = plotly.subplots.make_subplots

    if tokens:
        length = df['belief_tokens']
        history_length = df['history_tokens']
    else:
        length = df['belief'].apply(lambda d: len(str(d)))
        history_length = df['data'].apply(lambda d: len(str(d.get('conversation', [])[2:])))

    df2 = df.assign(_belief_len=length, _history_len=history_length)
    agg = df2.groupby(['env','model','attempt'])._belief_len.agg(['mean','std','count']).reset_index()
    agg['sem'] = agg['std'] / agg['count'].apply(lambda n: sqrt(n) if n > 0 else 1)

    colors = [
        "#FF6347",  # red
        "#4682B4",  # blue
        "#32CD32",  # green
    ]
    if tokens:
        metric = 'tokens'
    else:
        metric = 'chars'

    fig = make_subplots(rows=1, cols=len(envs), shared_yaxes=False, subplot_titles=envs)
    
    # Set y-axis limits for each subplot
    ylims = [[0,1000], [0,400],[0,650],[0,900],[0,350],[0,1300]]
    if not tokens:
        ylims = np.array(ylims)*4
        
    for col_idx in range(1, len(envs) + 1):
        fig.update_yaxes(range=ylims[col_idx-1], row=1, col=col_idx)

    model_color_map = {
        'Deepseek V3 (ABBEL)': '#4682B4',
        'Deepseek R1 (ABBEL)': '#FF6347',
        'Gemini 2.5 Pro (ABBEL)': '#32CD32',
    }

    models_seen = set()
    color_idx = 0
    for col_idx, env in enumerate(envs, start=1):

        # Plot the average history length for each env in gray (across all models and games, per step)
        sub_hist = df2[df2['env'] == env].groupby('attempt')['_history_len'].agg(['mean', 'std', 'count']).reset_index()
        sub_hist['sem'] = sub_hist['std'] / sub_hist['count'].apply(lambda n: sqrt(n) if n > 0 else 1)

         # Add shaded region for error bounds
        error_lower = sub_hist['mean'] - sub_hist['sem']
        error_upper = sub_hist['mean'] + sub_hist['sem']
        x = list(sub_hist['attempt'])
        # Add the lower error bound trace
        fig.add_trace(go.Scatter(
            x=x,
            y=error_lower,
            mode='lines',
            line=dict(width=0), # Hide the line for the lower bound
            showlegend=False,
            hoverinfo="skip",
            legendgroup='history'
        ),row=1, col=col_idx)

        # Add the upper error bound trace and fill to the lower bound
        fig.add_trace(go.Scatter(
            x=x,
            y=error_upper,
            mode='lines',
            fill='tonexty', # Fill the area between this trace and the previous one
            fillcolor='rgba(100,100,100,0.2)',
            line=dict(width=0), # Hide the line for the upper bound
            hoverinfo="skip",
            showlegend=False,
            legendgroup='history'
        ),
        row=1, col=col_idx)
        
        # Add main history line
        fig.add_trace(
            go.Scatter(
                x=sub_hist['attempt'],
                y=sub_hist['mean'],
                mode='lines+markers',
                name='Mean history length',
                legendgroup='history',
                showlegend=(col_idx == 1),  # only show legend once
                line=dict(color='gray', width=2, dash='dot'),
                marker=dict(color='gray')
            ),
            row=1, col=col_idx
        )

        sub = agg[agg['env'] == env]
        for model in sub['model'].unique():
            mdf = sub[sub['model'] == model].sort_values('attempt')
            if model not in model_color_map:
                model_color_map[model] = colors[color_idx % len(colors)]
                color_idx += 1
            show_legend = model not in models_seen
            models_seen.add(model)
        
            # Add shaded region for error bounds
            error_lower = mdf['mean'] - mdf['sem']
            error_upper = mdf['mean'] + mdf['sem']
            x = list(mdf['attempt'])
            # Add the lower error bound trace
            fig.add_trace(go.Scatter(
                x=x,
                y=error_lower,
                mode='lines',
                line=dict(width=0), # Hide the line for the lower bound
                showlegend=False,
                hoverinfo="skip",
                legendgroup=str(model)
            ),row=1, col=col_idx)

            # Add the upper error bound trace and fill to the lower bound
            fig.add_trace(go.Scatter(
                x=x,
                y=error_upper,
                mode='lines',
                fill='tonexty', # Fill the area between this trace and the previous one
                fillcolor=f"rgba({int(model_color_map[model][1:3], 16)}, {int(model_color_map[model][3:5], 16)}, {int(model_color_map[model][5:7], 16)}, 0.2)",
                line=dict(width=0), # Hide the line for the upper bound
                hoverinfo="skip",
                showlegend=False,
                legendgroup=str(model)
            ),
            row=1, col=col_idx)
            
            # Add main line
            fig.add_trace(
                go.Scatter(
                    x=mdf['attempt'],
                    y=mdf['mean'],
                    mode='lines+markers',
                    name=str(model.split('(')[0]),
                    legendgroup=str(model),
                    showlegend=show_legend,
                    line=dict(color=model_color_map[model], width=2)
                ),
                row=1, col=col_idx
            )

        fig.update_xaxes(
            title_text='Step',
            title_standoff=0,
            color='black',  # black axis
            showline=True,
            linecolor='black',
            linewidth=1,
            showgrid=False,
            row=1, col=col_idx
        )

        fig.update_yaxes(
            title_text=f'Mean Belief Length ({metric})' if col_idx == 1 else None,
            showline=True,
            linecolor='black',
            linewidth=1,
            tickfont=dict(color='black'),
            showgrid=True,
            gridcolor='lightgray',
            row=1, col=col_idx
        )

    fig.update_layout(
        width=1200,
        height=400,
        paper_bgcolor='white',
        plot_bgcolor='white',
        font=dict(family='Times New Roman, serif', size=14),
        legend=dict(
            orientation='v',   # vertical
            yanchor='middle',
            y=0.5,
            xanchor='left',
            x=1.02,
            bordercolor='black',
            borderwidth=1,
            bgcolor='rgba(255,255,255,0.9)'
        )
    )

    return fig


In [10]:
fig = plot_belief_length(df_belief, tokens=True)
fig.show()

In [11]:
fig.write_image('figures/frontier_belief_tokens.pdf', width=1200, height=350)

## 4. Greediness Metrics

In [23]:
df_mastermind = df[df['env'] == 'Mastermind']
df_wordle = df[df['env'] == 'Wordle']

In [29]:
df_wordle.loc[:, 'guess'] = df_wordle['guess'].apply(lambda x: ' '.join(list(str(x))))

In [24]:
df_wordle

,model,game_id,env,attempt,info,raw_guess,guess,response,word_limit,score,done,data,belief,action_reasoning,belief_reasoning,model_info
6,Deepseek V3 (belief prompting),1,Wordle,0,both,"<Answer> CRANE </Answer> \n\nReasoning: ""CRANE...",CRANE,"First letter, c, is not in the target word \nS...",None,1.0,True,"{'agent_game_scenario': 'word', 'env_game_scen...",The target word is a five-letter word. \nThe ...,None,None,Deepseek V3 (belief prompting)
7,Deepseek V3 (belief prompting),1,Wordle,1,both,<Answer> PRISM </Answer> \n\nExplanation: Base...,PRISM,"First letter, p, is not in the target word \nS...",None,2.0,True,"{'agent_game_scenario': 'word', 'env_game_scen...",The target word is a five-letter word. \nThe ...,None,None,Deepseek V3 (belief prompting)
8,Deepseek V3 (belief prompting),1,Wordle,2,both,<Answer> GRASP </Answer>\n\nThis guess include...,GRASP,"First letter, g, is not in the target word \nS...",None,3.0,True,"{'agent_game_scenario': 'word', 'env_game_scen...",The target word is a five-letter word. \nThe ...,None,None,Deepseek V3 (belief prompting)
9,Deepseek V3 (belief prompting),1,Wordle,3,both,<Answer> TRUST </Answer> \n\nExplanation: Bas...,TRUST,Goal reached,None,4.0,True,"{'agent_game_scenario': 'word', 'env_game_scen...",The target word is a five-letter word. \nThe ...,None,None,Deepseek V3 (belief prompting)
10,Deepseek V3,1,Wordle,0,history,<Answer> CRANE </Answer>\n\nThis guess include...,CRANE,"First letter, c, is not in the target word \nS...",None,1.0,True,"{'agent_game_scenario': 'word', 'env_game_scen...",,None,,Deepseek V3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26614,Deepseek R1 (belief prompting),38,Wordle,1,both,<Answer> radio </Answer>,radio,"First letter, r, exists in the target word but...",None,2.0,True,"{'agent_game_scenario': 'word', 'env_game_scen...","Excluded letters: S, L, T, E, D, O \nIncluded...","Okay, let's figure out the next guess. The fir...","Okay, let's start by understanding the current...",Deepseek R1 (belief prompting)
26615,Deepseek R1 (belief prompting),38,Wordle,2,both,<Answer> prima </Answer>,prima,"First letter, p, is not in the target word \nS...",None,3.0,True,"{'agent_game_scenario': 'word', 'env_game_scen...","Excluded letters: S, L, T, E, D, O, P, M \nIn...","Okay, let's think through this step by step. S...","Okay, let's see. The user provided the current...",Deepseek R1 (belief prompting)
26616,Deepseek R1 (belief prompting),38,Wordle,3,both,<Answer> infra </Answer>,infra,"First letter, i, exists in the target word but...",None,4.0,True,"{'agent_game_scenario': 'word', 'env_game_scen...","Excluded letters: S, L, T, E, D, O, P, M, N, F...","Okay, let's see. So I'm on my fourth guess in ...","Okay, let me try to figure out how to update t...",Deepseek R1 (belief prompting)
26617,Deepseek R1 (belief prompting),38,Wordle,4,both,<Answer> virga </Answer>,virga,"First letter, v, is not in the target word \nS...",None,5.0,True,"{'agent_game_scenario': 'word', 'env_game_scen...","Excluded letters: S, L, T, E, D, O, P, M, N, F...","Okay, let's see. I'm on my fifth attempt in th...","Okay, let's tackle this Wordle problem step by...",Deepseek R1 (belief prompting)


In [32]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

def plot_action_repetition(df):
    """
    Create a bar chart showing greediness for each model.
    
    Greediness is defined as the fraction of times a model repeats
    a particular guess at a particular position across games.
    
    Parameters:
    df (pd.DataFrame): DataFrame with columns 'game_id', 'attempt', 'model_info', 'guess'
    
    Returns:
    plotly.graph_objects.Figure: Bar chart figure
    """
    
    # Fixed colors for different models
    colors = [
        "#ADD8E6",  # light blue
        "#4682B4",  # medium blue
        "#003366",  # dark blue
        "#FFB6C1",  # light red
        "#FF6347",  # medium red
        "#8B0000",  # dark red
        "#90EE90",  # light green
        "#32CD32",  # medium green
        "#006400",  # dark green
    ]
    
    greediness_results = []
    
    # Group by model_info
    for model_idx, (model, model_data) in enumerate(df.groupby('model_info')):
        # Group by game_id to get sequences of guesses
        game_sequences = []
        for game_id, game_data in model_data.groupby('game_id'):
            # Sort by attempt to get the correct sequence
            sequence = game_data.sort_values('attempt')['guess'].tolist()
            game_sequences.append(sequence)
        
        # Calculate greediness for each game separately to get variance
        game_greediness_values = []
        
        # For each game, calculate its greediness
        for seq in game_sequences:
            game_total_greediness = 0
            game_total_positions = 0
            
            # For each position in this game's sequence
            for pos in range(len(seq)):
                # Get all guesses at this position across ALL games for comparison
                guesses_at_position = []
                for other_seq in game_sequences:
                    if pos < len(other_seq):
                        guesses_at_position.append(other_seq[pos])
                
                if len(guesses_at_position) > 1:
                    # Count repeats
                    unique_guesses = len(set(guesses_at_position))
                    total_guesses = len(guesses_at_position)
                    
                    # Greediness = 1 - (unique_guesses / total_guesses)
                    position_greediness = 1 - (unique_guesses / total_guesses)
                    
                    game_total_greediness += position_greediness
                    game_total_positions += 1
            
            # Average greediness for this game
            if game_total_positions > 0:
                game_avg_greediness = (game_total_greediness / game_total_positions * 100)
                game_greediness_values.append(game_avg_greediness)
        
        # Calculate mean and standard error
        if game_greediness_values:
            avg_greediness = np.mean(game_greediness_values)
            std_error = np.std(game_greediness_values, ddof=1) / np.sqrt(len(game_greediness_values)) if len(game_greediness_values) > 1 else 0
        else:
            avg_greediness = 0
            std_error = 0
        
        greediness_results.append({
            'model': model,
            'greediness': avg_greediness,
            'std_error': std_error,
            'color': colors[model_idx % len(colors)]
        })

    desired_order = [0, 2, 1, 3, 5, 4, 6, 8, 7]
    greediness_results = [greediness_results[i] for i in desired_order]
    
    # Create the bar chart
    fig = go.Figure()
    
    # Add bars
    models = [result['model'] for result in greediness_results]
    greediness_values = [result['greediness'] for result in greediness_results]
    error_values = [result['std_error'] for result in greediness_results]
    bar_colors = [result['color'] for result in greediness_results]
    
    fig.add_trace(go.Bar(
        x=models,
        y=greediness_values,
        error_y=dict(
            type='data',
            array=error_values,
            visible=True,
            color='black',
            thickness=1.5,
            width=3
        ),
        marker_color=bar_colors,
        marker_line_color='black',
        marker_line_width=1,
        width=0.6
    ))
    
    # Update layout for professional appearance
    fig.update_layout(
        title={
            'text': '',
            'x': 0.5,
            'font': {'family': 'Computer Modern, serif', 'size': 16}
        },
        xaxis={
            'title': 'Model + Info (Wordle)',
            'title_font': {'family': 'Computer Modern, serif', 'size': 14},
            'tickfont': {'family': 'Computer Modern, serif', 'size': 14},
            'showgrid': False,
            'showline': True,
            'linewidth': 1,
            'linecolor': 'black',
            'mirror': True
        },
        yaxis={
            'title': 'Action Repetition',
            'title_font': {'family': 'Computer Modern, serif', 'size': 14},
            'tickfont': {'family': 'Computer Modern, serif', 'size': 14},
            'showgrid': False,
            'showline': True,
            'linewidth': 1,
            'linecolor': 'black',
            'mirror': True,
            'range': [0, max([g + e for g, e in zip(greediness_values, error_values)]) * 1.1 if greediness_values else 100]
        },
        plot_bgcolor='white',
        paper_bgcolor='white',
        font={'family': 'Computer Modern, serif'},
        width=800,
        height=500,
        margin=dict(l=80, r=50, t=80, b=80)
    )
    
    # Add grid lines manually for a cleaner look
    fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor='lightgray')
    
    return fig

In [33]:
rep = plot_action_repetition(df_mastermind)
rep.show()

In [15]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

def plot_greediness(df):
    """
    Create a bar chart showing greediness for each model.

    Greediness is defined as the fraction of times the correct guess is made
    for any position (average across rows and positions).

    Parameters:
    df (pd.DataFrame): DataFrame with columns 'game_id', 'attempt', 'model_info', 'guess'

    Returns:
    plotly.graph_objects.Figure: Bar chart figure
    """

    # Fixed colors for different models
    colors = [
        "#ADD8E6",  # light blue
        "#4682B4",  # medium blue
        "#003366",  # dark blue
        "#FFB6C1",  # light red
        "#FF6347",  # medium red
        "#8B0000",  # dark red
        "#90EE90",  # light green
        "#32CD32",  # medium green
        "#006400",  # dark green
    ]

    greediness_results = []

    # Group by model_info
    for model_idx, (model, model_data) in enumerate(df.groupby('model_info')):
        per_row_fractions = []
        for row_id in range(len(model_data)):
            # Get target and guess
            # Fix: check if 'data' is a dict and contains 'env_game_scenario'
            data_field = model_data.iloc[row_id]['data']
            if isinstance(data_field, dict) and 'env_game_scenario' in data_field:
                target = str(data_field['env_game_scenario'])
            else:
                # fallback: skip this row if not present
                continue
            guess_str = str(model_data.iloc[row_id]['guess'])
            guess = guess_str.split(' ')
            # Only compare up to the length of the shorter of guess/target
            n_positions = min(len(target), len(guess))
            if n_positions == 0:
                continue
            correct = 0
            for idx in range(n_positions):
                if str(guess[idx]) == str(target[idx]):
                    correct += 1
            per_row_fractions.append(correct / n_positions)
        if len(per_row_fractions) == 0:
            avg_greediness = 0.0
            std_error = 0.0
        else:
            avg_greediness = np.mean(per_row_fractions)
            std_error = np.std(per_row_fractions, ddof=1) / np.sqrt(len(per_row_fractions)) if len(per_row_fractions) > 1 else 0.0
        greediness_results.append({
            'model': model,
            'greediness': avg_greediness,
            'std_error': std_error,
            'color': colors[model_idx % len(colors)]
        })

    # Optionally reorder for display
    desired_order = [0, 2, 1, 3, 5, 4, 6, 8, 7]
    if len(greediness_results) == len(desired_order):
        greediness_results = [greediness_results[i] for i in desired_order]

    # Create the bar chart
    fig = go.Figure()

    # Add bars
    models = [result['model'] for result in greediness_results]
    greediness_values = [result['greediness'] for result in greediness_results]
    error_values = [result['std_error'] for result in greediness_results]
    bar_colors = [result['color'] for result in greediness_results]

    fig.add_trace(go.Bar(
        x=models,
        y=greediness_values,
        error_y=dict(
            type='data',
            array=error_values,
            visible=True,
            color='black',
            thickness=1.5,
            width=3
        ),
        marker_color=bar_colors,
        marker_line_color='black',
        marker_line_width=1,
        width=0.6
    ))

    # Update layout for professional appearance
    fig.update_layout(
        title={
            'text': '',
            'x': 0.5,
            'font': {'family': 'Computer Modern, serif', 'size': 16}
        },
        xaxis={
            'title': 'Model + Info (Wordle)',
            'title_font': {'family': 'Computer Modern, serif', 'size': 14},
            'tickfont': {'family': 'Computer Modern, serif', 'size': 14},
            'showgrid': False,
            'showline': True,
            'linewidth': 1,
            'linecolor': 'black',
            'mirror': True
        },
        yaxis={
            'title': 'Greediness (Correct)',
            'title_font': {'family': 'Computer Modern, serif', 'size': 14},
            'tickfont': {'family': 'Computer Modern, serif', 'size': 14},
            'showgrid': False,
            'showline': True,
            'linewidth': 1,
            'linecolor': 'black',
            'mirror': True,
            'range': [0, max([g + e for g, e in zip(greediness_values, error_values)]) * 1.1 if greediness_values else 100]
        },
        plot_bgcolor='white',
        paper_bgcolor='white',
        font={'family': 'Computer Modern, serif'},
        width=800,
        height=500,
        margin=dict(l=80, r=50, t=80, b=80)
    )

    # Add grid lines manually for a cleaner look
    fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor='lightgray')

    return fig

In [16]:
models_list = [
    'deepseek/deepseek-chat (belief)',
    'deepseek/deepseek-chat (history)',
    'deepseek/deepseek-chat (both)',
    'deepseek/deepseek-r1 (belief)'
    'deepseek/deepseek-r1 (history)',
    'deepseek/deepseek-r1 (both)',
    'google/gemini-2.5-pro (belief)',
    'google/gemini-2.5-pro (history)',
    'google/gemini-2.5-pro (both)',
    ]

In [30]:
#fig_ar_mm = plot_action_repetition(df_mastermind)
fig_gr_mm = plot_greediness(df_mastermind)
#fig_ar_wd = plot_action_repetition(df_wordle)
fig_gr_wd = plot_greediness(df_wordle)

In [28]:
fig_gr_mm.show()


In [31]:
fig_gr_wd.show()

In [114]:
list(df.model_info.unique())

['deepseek/deepseek-chat (history)',
 'deepseek/deepseek-chat (both)',
 'google/gemini-2.5-pro (history)',
 'deepseek/deepseek-chat (belief)',
 'deepseek/deepseek-r1 (history)',
 'google/gemini-2.5-pro (belief)',
 'google/gemini-2.5-pro (both)',
 'deepseek/deepseek-r1 (both)',
 'deepseek/deepseek-r1 (belief)']

## 5. Sizes of Reasoning Traces

In [146]:
# df_reasoning = df[(~df['model'].str.contains('deepseek/deepseek-chat')) & (df['info'] == 'belief')]
# df_reasoning = df[
#     (~df['model'].str.contains('deepseek/deepseek-chat')) &
#     (df['info'].isin(['belief', 'history']))
# ]
df_reasoning = df[~df['model'].str.contains('V3')]
len(df_reasoning), len(df)

(17283, 27106)

In [147]:
# Convert dataframe to huggingface dataset
dataset = Dataset.from_pandas(df_reasoning)

def batch_count_belief_reasoning_tokens(batch):
    results = []
    for i, model in enumerate(batch['model']):
        model_key = model.split(' ')[1]
        text = batch['belief_reasoning'][i]
        token_count = token_counter_dict[model_key](text)
        results.append(token_count)
    return {'belief_reasoning_tokens': results}

def batch_count_action_reasoning_tokens(batch):
    results = []
    for i, model in enumerate(batch['model']):
        model_key = model.split(' ')[1]
        text = batch['action_reasoning'][i]
        token_count = token_counter_dict[model_key](text)
        results.append(token_count)
    return {'action_reasoning_tokens': results}

# Batch encode belief tokens
dataset = dataset.map(batch_count_belief_reasoning_tokens, batched=True, batch_size=100)

print("===============================/n calculating action reasoning tokens now!")

# Batch encode history tokens
dataset = dataset.map(batch_count_action_reasoning_tokens, batched=True, batch_size=100)

# Convert back to pandas and update original dataframe
df_reasoning['belief_reasoning_tokens'] = dataset['belief_reasoning_tokens']
df_reasoning['action_reasoning_tokens'] = dataset['action_reasoning_tokens']

Map:   0%|          | 0/17283 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (16886 > 16384). Running this sequence through the model will result in indexing errors


===============================/n calculating action reasoning tokens now!


Map:   0%|          | 0/17283 [00:00<?, ? examples/s]

/tmp/ipykernel_3616647/1316764292.py:31: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_3616647/1316764292.py:32: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [166]:
df_reasoning['history'] = df_reasoning['data'].apply(lambda d: str(d.get('conversation', [])[2:]))

# Convert dataframe to huggingface dataset
dataset = Dataset.from_pandas(df_reasoning)

def batch_count_belief_tokens(batch):
    results = []
    for i, model in enumerate(batch['model']):
        model_key = model.split(' ')[1]
        belief_text = batch['belief'][i]
        token_count = token_counter_dict[model_key](belief_text)
        results.append(token_count)
    return {'belief_tokens': results}

def batch_count_history_tokens(batch):
    results = []
    for i, model in enumerate(batch['model']):
        model_key = model.split(' ')[1]
        history_text = batch['history'][i]
        token_count = token_counter_dict[model_key](history_text)
        results.append(token_count)
    return {'history_tokens': results}

def batch_count_x_tokens(batch, key='key'):
    results = []
    for i, model in enumerate(batch['model']):
        model_key = model.split(' ')[1]
        text = batch[key][i]
        token_count = token_counter_dict[model_key](text)
        results.append(token_count)
    return {key+'_tokens': results}

# Batch encode belief tokens
dataset = dataset.map(batch_count_x_tokens, batched=True, batch_size=100, fn_kwargs={"key": 'belief'})

# Batch encode history tokens
dataset = dataset.map(batch_count_x_tokens, batched=True, batch_size=100, fn_kwargs={"key": 'history'})

dataset = dataset.map(batch_count_x_tokens, batched=True, batch_size=100, fn_kwargs={"key": 'guess'})
dataset = dataset.map(batch_count_x_tokens, batched=True, batch_size=100, fn_kwargs={"key": 'response'})
dataset = dataset.map(batch_count_x_tokens, batched=True, batch_size=100, fn_kwargs={"key": 'raw_guess'})


# Convert back to pandas and update original dataframe
df_reasoning['belief_tokens'] = dataset['belief_tokens']
df_reasoning['history_tokens'] = dataset['history_tokens']
df_reasoning['guess_tokens'] = dataset['guess_tokens']
df_reasoning['response_tokens'] = dataset['response_tokens']
df_reasoning['raw_guess_tokens'] = dataset['raw_guess_tokens']

/tmp/ipykernel_3616647/3855247512.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



Map:   0%|          | 0/17283 [00:00<?, ? examples/s]

Map:   0%|          | 0/17283 [00:00<?, ? examples/s]

Map:   0%|          | 0/17283 [00:00<?, ? examples/s]

Map:   0%|          | 0/17283 [00:00<?, ? examples/s]

/tmp/ipykernel_3616647/3855247512.py:43: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_3616647/3855247512.py:44: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_3616647/3855247512.py:45: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ip

In [154]:
def plot_reasoning_length(df, reasoning_types = ['Belief Reasoning', 'Action Reasoning'], tokens=True):
    envs = envs_ordered
    go = plotly.graph_objects
    make_subplots = plotly.subplots.make_subplots

    # Compute lengths for both reasoning fields
    if tokens:
        belief_reasoning_length = df['belief_reasoning_tokens']
        action_reasoning_length = df['action_reasoning_tokens']
        metric = 'tokens'
    else:
        belief_reasoning_length = df['belief_reasoning'].astype(str).str.len()
        action_reasoning_length = df['action_reasoning'].astype(str).str.len()
        metric = 'chars'

    # Prepare dataframes for aggregation
    df_belief = df.assign(_reasoning_len=belief_reasoning_length, _reasoning_type='Belief Reasoning')
    df_action = df.assign(_reasoning_len=action_reasoning_length, _reasoning_type='Action Reasoning')
    df_total = df.assign(_reasoning_len=belief_reasoning_length + action_reasoning_length, _reasoning_type='Total Reasoning')
    df2 = pd.concat([df_belief, df_action, df_total], ignore_index=True)

    agg = (
        df2.groupby(['env', 'model', 'attempt', '_reasoning_type'])
        ._reasoning_len.agg(['mean', 'std', 'count'])
        .reset_index()
    )
    agg['sem'] = agg['std'] / agg['count'].apply(lambda n: sqrt(n) if n > 0 else 1)

    colors = [
        "#8B0000",  # dark red
        "#FF6347",  # medium red
        "#FFB6C1",  # light red
        "#90EE90",  # light green
        "#006400",  # dark green
        "#32CD32",  # medium green
        "#ADD8E6",  # light blue
        "#4682B4",  # medium blue
        "#003366",  # dark blue
    ]

    # Two rows: 1 for belief_reasoning, 2 for action_reasoning, but reduce vertical space and share axes
    fig = make_subplots(
        rows=2,
        cols=len(envs),
        shared_yaxes=False,
        shared_xaxes=False,
        subplot_titles=[f"{env}" for env in envs],
        row_titles=["", ""],
        vertical_spacing=0.1 
    )

    # model_color_map = {}

    model_color_map = {
        'Deepseek V3 (ABBEL)': '#4682B4',
        'Deepseek R1 (ABBEL)': '#8B0000',
        'Gemini 2.5 Pro (ABBEL)': '#006400',
        'Deepseek V3': '#4682B4',
        'Deepseek R1': '#FFB6C1',
        'Gemini 2.5 Pro': '#90EE90',
        'Deepseek V3 (belief prompting)': '#4682B4',
        'Deepseek R1 (belief prompting)': '#FF6347',
        'Gemini 2.5 Pro (belief prompting)': '#32CD32',
    }

    models_seen = set()
    color_idx = 0

    for col_idx, env in enumerate(envs, start=1):
        for row_idx, reasoning_type in enumerate(reasoning_types, start=1):
            sub = agg[(agg['env'] == env) & (agg['_reasoning_type'] == reasoning_type)]
            for model in sub['model'].unique():
                mdf = sub[sub['model'] == model].sort_values('attempt')
                if model not in model_color_map:
                    model_color_map[model] = colors[color_idx % len(colors)]
                    color_idx += 1

                # if ('V3' not in model) and not ('Belief' in reasoning_type and '(' not in model):
                if not ('Belief' in reasoning_type and '(' not in model):
                    show_legend = model not in models_seen
                    models_seen.add(model)

                    # Add shaded region for error bounds
                    error_lower = mdf['mean'] - mdf['sem']
                    error_upper = mdf['mean'] + mdf['sem']
                    x = list(mdf['attempt'])
                    # Add the lower error bound trace
                    fig.add_trace(go.Scatter(
                        x=x,
                        y=error_lower,
                        mode='lines',
                        line=dict(width=0), # Hide the line for the lower bound
                        showlegend=False,
                        hoverinfo="skip",
                        legendgroup=str(model)
                    ),row=row_idx, col=col_idx)

                    # Add the upper error bound trace and fill to the lower bound
                    fig.add_trace(go.Scatter(
                        x=x,
                        y=error_upper,
                        mode='lines',
                        fill='tonexty', # Fill the area between this trace and the previous one
                        fillcolor=f"rgba{tuple(list(plotly.colors.hex_to_rgb(model_color_map[model])) + [0.2])}",
                        line=dict(width=0), # Hide the line for the upper bound
                        hoverinfo="skip",
                        showlegend=False,
                        legendgroup=str(model)
                    ),
                    row=row_idx, col=col_idx)
                    
                    # Add main line
                    fig.add_trace(
                        go.Scatter(
                            x=mdf['attempt'],
                            y=mdf['mean'],
                            mode='lines+markers',
                            name=str(model),
                            legendgroup=str(model),
                            showlegend=show_legend,
                            line=dict(color=model_color_map[model], width=2)
                        ),
                        row=row_idx, col=col_idx
                    )

            fig.update_xaxes(
                title_text='Step' if row_idx == 2 else '',
                color='black',
                showline=True,
                linecolor='black',
                linewidth=1,
                showgrid=False,
                row=row_idx, col=col_idx
            )

            fig.update_yaxes(
                title_text=f'{reasoning_type} ({metric})' if col_idx == 1 else None,
                showline=True,
                linecolor='black',
                linewidth=1,
                tickfont=dict(color='black'),
                showgrid=True,
                gridcolor='lightgray',
                row=row_idx, col=col_idx
            )

        # Layout with horizontal legend
    fig.update_layout(
        height=700,
        # width=fig_width,
        template='simple_white',
        font=dict(family='Computer Modern, serif', size=16),
        barmode='group',
        showlegend=True,
        legend=dict(
            # title={'text': ' Model (info)'},
            orientation='v',
            yanchor='middle',
            y=0.5,
            xanchor='left',
            x=1.02,
            bgcolor='rgba(255,255,255,0.9)',
            bordercolor='black',
            borderwidth=1,
            traceorder='grouped',
            itemsizing='constant'
        ),
        margin=dict(t=100, b=50, l=80, r=80),
        plot_bgcolor='white'
    )
    
    # Reorder legend items
    legend_order = ['Deepseek R1', 'Deepseek R1 (belief prompting)', 'Deepseek R1 (ABBEL)', 
                   'Gemini 2.5 Pro', 'Gemini 2.5 Pro (belief prompting)', 'Gemini 2.5 Pro (ABBEL)']
    
    # Get current traces and reorder them
    traces = list(fig.data)
    reordered_traces = []
    
    # First add traces in the desired legend order
    for model_name in legend_order:
        for trace in traces:
            if hasattr(trace, 'name') and trace.name == model_name:
                reordered_traces.append(trace)
    
    # Add any remaining traces that weren't in the legend order
    for trace in traces:
        if trace not in reordered_traces:
            reordered_traces.append(trace)
    
    # Update the figure with reordered traces
    fig.data = reordered_traces

    return fig


In [155]:
fig = plot_reasoning_length(df_reasoning, reasoning_types = ['Belief Reasoning', 'Total Reasoning'], tokens=False)
fig.show()

In [156]:
fig = plot_reasoning_length(df_reasoning, reasoning_types = ['Belief Reasoning', 'Total Reasoning'], tokens=True)
fig.show()

In [157]:
fig.write_image('figures/frontier_belief_total_reasoning_tokens.pdf', width=1500, height=600)

In [94]:
action_reasoning_length = df_reasoning['action_reasoning'].astype(str).str.len()
print(action_reasoning_length.iloc[0])
belief_length = df_reasoning['belief'].astype(str).str.len()
history_length = df_reasoning['data'].apply(lambda d: len(str(d.get('conversation', [])[2:])))
action_reasoning_length += history_length*(df_reasoning['model'].apply(lambda x: 'ABBEL' not in x))
action_reasoning_length.iloc[0]

551


np.int64(1180)

In [7]:
df_reasoning['belief_length'] = df_reasoning['belief'].astype(str).str.len()
df_reasoning['prev_belief_len'] = df_reasoning.groupby(['model', 'game_id', 'env'])['belief_length'].shift(1).fillna(0) * (df_reasoning['attempt'] != 0)

/tmp/ipykernel_2296185/235403059.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_reasoning['belief_length'] = df_reasoning['belief'].astype(str).str.len()
/tmp/ipykernel_2296185/235403059.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_reasoning['prev_belief_len'] = df_reasoning.groupby(['model', 'game_id', 'env'])['belief_length'].shift(1).fillna(0) * (df_reasoning['attempt'] != 0)


In [8]:
def get_context_reasoning_lengths(df, include_context=True):
    df['belief_reasoning_length'] = df['belief_reasoning'].astype(str).str.len()
    df['action_reasoning_length'] = df['action_reasoning'].astype(str).str.len()
    if include_context:
        df['belief_length'] = df['belief'].astype(str).str.len()
        df['history_length'] = df['data'].apply(lambda d: len(str(d.get('conversation', [])[2:])))
        df['prev_belief_len'] = df.groupby(['model', 'game_id', 'env'])['belief_length'].shift(1).fillna(0) * (df['attempt'] != 0)
        df['prev_history_len'] = df.groupby(['model', 'game_id', 'env'])['history_length'].shift(1).fillna(0) * (df['attempt'] != 0)
        df['belief_context_len'] = df['guess'].astype(str).str.len() + df['response'].astype(str).str.len() + df['prev_belief_len'] + df['prev_history_len']*(df['model'].apply(lambda x: 'belief prompting' in x))
        df['belief_context_len'] = df['belief_context_len']*(df['model'].apply(lambda x: 'belief prompting' in x or 'ABBEL' in x))
        df['action_context_len'] = df['prev_belief_len'] + df['history_length']*(df['model'].apply(lambda x: 'ABBEL' not in x))
    return df

df_reasoning = get_context_reasoning_lengths(df_reasoning, include_context=True)

/tmp/ipykernel_2296185/1863337873.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['belief_reasoning_length'] = df['belief_reasoning'].astype(str).str.len()
/tmp/ipykernel_2296185/1863337873.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['action_reasoning_length'] = df['action_reasoning'].astype(str).str.len()
/tmp/ipykernel_2296185/1863337873.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instea

In [163]:
df_reasoning[
    (df_reasoning['model'] == 'Deepseek R1 (belief prompting)') & \
        (df_reasoning['env'] == 'Wordle') & \
            (df_reasoning['game_id'] == str(1))
            ]

,model,game_id,env,attempt,info,raw_guess,guess,response,word_limit,score,...,belief_reasoning,model_info,belief_length,prev_belief_len,history_length,prev_history_len,belief_context_len,action_context_len,belief_reasoning_length,action_reasoning_length
1600,Deepseek R1 (belief prompting),1,Wordle,0,both,\n\n<Answer> crane </Answer>,crane,"first letter, c, is not in the target word \ns...",None,1.0,...,"Okay, let's see. The user is playing Wordle an...",Deepseek R1 (belief prompting),403,0.0,661,0.0,285.0,661.0,2857,806
1601,Deepseek R1 (belief prompting),1,Wordle,1,both,<Answer> baste </Answer>,baste,"first letter, b, is not in the target word \ns...",None,2.0,...,"Okay, so I need to update the belief based on ...",Deepseek R1 (belief prompting),315,403.0,1358,661.0,1385.0,1761.0,16506,19274
1602,Deepseek R1 (belief prompting),1,Wordle,2,both,<Answer> plate </Answer>,plate,"first letter, p, is not in the target word \ns...",None,3.0,...,"Okay, let's see. The initial belief says that ...",Deepseek R1 (belief prompting),408,315.0,2051,1358.0,1990.0,2366.0,5661,14380
1603,Deepseek R1 (belief prompting),1,Wordle,3,both,<Answer> haute </Answer>,haute,goal reached,None,4.0,...,"Okay, let me try to figure out how to update t...",Deepseek R1 (belief prompting),222,408.0,2440,2051.0,2476.0,2848.0,9885,22443


In [177]:
def plot_reasoning_length_one_fig(df, reasoning_type='Belief Reasoning', title='Belief Reasoning Length', tokens=True, total_type='total reasoning'):
    envs = envs_ordered
    go = plotly.graph_objects
    make_subplots = plotly.subplots.make_subplots

    # Compute lengths for both reasoning fields
    if tokens:
        df.loc[:, 'belief_reasoning_length'] = df['belief_reasoning_tokens']
        df.loc[:, 'action_reasoning_length'] = df['action_reasoning_tokens']
        df.loc[:, 'guess_length'] = df['guess_tokens']
        df.loc[:, 'raw_guess_length'] = df['raw_guess_tokens']
        df.loc[:, 'response_length'] = df['response_tokens']
        df.loc[:, 'belief_length'] = df['belief_tokens']
        df.loc[:, 'history_length'] = df['history_tokens']
    else:
        df.loc[:, 'belief_reasoning_length'] = df.loc[:, 'belief_reasoning'].astype(str).str.len()
        df.loc[:, 'action_reasoning_length'] = df.loc[:, 'action_reasoning'].astype(str).str.len()
        df.loc[:, 'guess_length'] = df.loc[:, 'guess'].astype(str).str.len()
        df.loc[:, 'raw_guess_length'] = df.loc[:, 'raw_guess'].astype(str).str.len()
        df.loc[:, 'response_length'] = df.loc[:, 'response'].astype(str).str.len()
        df.loc[:, 'belief_length'] = df.loc[:, 'belief'].astype(str).str.len()
        df.loc[:, 'history_length'] = df.loc[:, 'data'].apply(lambda d: len(str(d.get('conversation', [])[2:])))
    if total_type in ('processed per turn', 'memory usage'):
        df.loc[:, 'prev_belief_len'] = df.groupby(['model', 'game_id', 'env'])['belief_length'].shift(1).fillna(0) * (df['attempt'] != 0)
        df.loc[:, 'prev_history_len'] = df.groupby(['model', 'game_id', 'env'])['history_length'].shift(1).fillna(0) * (df['attempt'] != 0)
        df.loc[:, 'belief_context_len'] = df['prev_history_len']*(df['model'].apply(lambda x: 'belief prompting' in x)) + df['prev_belief_len'] + df['guess_length'] + df['response_length'] 
        df.loc[:, 'belief_context_len'] = df['belief_context_len']*(df['model'].apply(lambda x: 'belief prompting' in x or 'ABBEL' in x)) # ensure belief context len is 0 for vanilla
        df.loc[:, 'action_context_len'] = df['history_length']*(df['model'].apply(lambda x: 'ABBEL' not in x)) + df['prev_belief_len'] # prev belief len is always 0 for vanilla

    # Prepare dataframes for aggregation
    df_belief = df.assign(_reasoning_len=df['belief_reasoning_length'], _reasoning_type='Belief Reasoning')
    df_action = df.assign(_reasoning_len=df['action_reasoning_length'], _reasoning_type='Action Reasoning')
    if total_type == 'processed per turn':
        df_total = df.assign(_reasoning_len=df['belief_context_len']+df['belief_reasoning_length'] + df['belief_length'] + df['action_context_len']+df['action_reasoning_length']+df['raw_guess_length'], _reasoning_type='Total Reasoning') # This is the total tokens/characters processed per turn, combining belief update step and action selection step
    elif total_type == 'memory usage':
        df_total = df.assign(_reasoning_len=np.maximum(df['belief_context_len']+df['belief_reasoning_length'] + df['belief_length'],df['action_context_len']+df['action_reasoning_length']+df['raw_guess_length']), _reasoning_type='Total Reasoning') # this is the memory required per turn, as the context from the belief step is thrown away for the action step
    elif total_type == 'total reasoning':
        df_total = df.assign(_reasoning_len=df['belief_reasoning_length'] + df['action_reasoning_length'], _reasoning_type='Total Reasoning')
    else:
        raise ValueError(f"Total type {total_type} not supported")
    df2 = pd.concat([df_belief, df_action, df_total], ignore_index=True)

    agg = (
        df2.groupby(['env', 'model', 'attempt', '_reasoning_type'])
        ._reasoning_len.agg(['mean', 'std', 'count'])
        .reset_index()
    )
    agg['sem'] = agg['std'] / agg['count'].apply(lambda n: sqrt(n) if n > 0 else 1)

    colors = [
        "#8B0000",  # dark red
        "#FF6347",  # medium red
        "#FFB6C1",  # light red
        "#90EE90",  # light green
        "#006400",  # dark green
        "#32CD32",  # medium green
        "#ADD8E6",  # light blue
        "#4682B4",  # medium blue
        "#003366",  # dark blue
    ]

    model_color_map = {
        'Deepseek V3 (ABBEL)': '#4682B4',
        'Deepseek R1 (ABBEL)': '#8B0000',
        'Gemini 2.5 Pro (ABBEL)': '#006400',
        'Deepseek V3': '#4682B4',
        'Deepseek R1': '#FFB6C1',
        'Gemini 2.5 Pro': '#90EE90',
        'Deepseek V3 (belief prompting)': '#4682B4',
        'Deepseek R1 (belief prompting)': '#FF6347',
        'Gemini 2.5 Pro (belief prompting)': '#32CD32',
    }
    
    # Create separate figure for Belief Reasoning
    fig = make_subplots(
        rows=1,
        cols=len(envs),
        shared_yaxes=False,
        shared_xaxes=False,
        subplot_titles=[f"{env}" for env in envs],
        horizontal_spacing=0.03
    )

    models_seen = set()
    color_idx = 0

    for col_idx, env in enumerate(envs, start=1):
        sub = agg[(agg['env'] == env) & (agg['_reasoning_type'] == reasoning_type)]
        if total_type in ('processed per turn', 'memory usage'):
            sub = sub[sub['model'].str.contains('R1')]
        for model in sub['model'].unique():
            mdf = sub[sub['model'] == model].sort_values('attempt')
            if model not in model_color_map:
                model_color_map[model] = colors[color_idx % len(colors)]
                color_idx += 1

            if '(' in model or (reasoning_type != 'Belief Reasoning'):
                show_legend = model not in models_seen
                models_seen.add(model)
                # Add shaded region for error bounds
                error_lower = mdf['mean'] - mdf['sem']
                error_upper = mdf['mean'] + mdf['sem']
                x = list(mdf['attempt'])
                # Add the lower error bound trace
                fig.add_trace(go.Scatter(
                    x=x,
                    y=error_lower,
                    mode='lines',
                    line=dict(width=0), # Hide the line for the lower bound
                    showlegend=False,
                    hoverinfo="skip",
                    legendgroup=str(model)
                ),row=1, col=col_idx)

                # Add the upper error bound trace and fill to the lower bound
                fig.add_trace(go.Scatter(
                    x=x,
                    y=error_upper,
                    mode='lines',
                    fill='tonexty', # Fill the area between this trace and the previous one
                    fillcolor=f"rgba{tuple(list(plotly.colors.hex_to_rgb(model_color_map[model])) + [0.2])}",
                    line=dict(width=0), # Hide the line for the upper bound
                    hoverinfo="skip",
                    showlegend=False,
                    legendgroup=str(model)
                ),
                row=1, col=col_idx)
                
                
                # Add main line
                fig.add_trace(
                    go.Scatter(
                        x=mdf['attempt'],
                        y=mdf['mean'],
                        mode='lines+markers',
                        name=str(model),
                        legendgroup=str(model),
                        showlegend=show_legend,
                        line=dict(color=model_color_map[model], width=2)
                    ),
                    row=1, col=col_idx
                )

        fig.update_xaxes(
            title_text='Step',
            title_standoff=0,
            color='black',
            showline=True,
            linecolor='black',
            linewidth=1,
            showgrid=False,
            row=1, col=col_idx
        )

        fig.update_yaxes(
            title_text=title if col_idx == 1 else None,
            showline=True,
            linecolor='black',
            linewidth=1,
            tickfont=dict(color='black'),
            showgrid=True,
            gridcolor='lightgray',
            row=1, col=col_idx
        )

    fig.update_layout(
        height=350,
        template='simple_white',
        font=dict(family='Computer Modern, serif', size=16),
        barmode='group',
        showlegend=True,
        legend=dict(
            orientation='v',
            yanchor='middle',
            y=0.5,
            xanchor='left',
            x=1.02,
            bgcolor='rgba(255,255,255,0.9)',
            bordercolor='black',
            borderwidth=1
        ),
        #margin=dict(t=100, b=50, l=80, r=80),
        plot_bgcolor='white'
    )
    
    return fig

In [179]:
fig = plot_reasoning_length_one_fig(df_reasoning, reasoning_type='Action Reasoning', title='Action Reasoning Tokens', tokens=True)
fig.show()

In [165]:
fig.write_image('figures/frontier_action_reasoning_tokens.pdf', width=1500, height=390)

In [185]:
fig = plot_reasoning_length_one_fig(df_reasoning, reasoning_type='Total Reasoning', title='Memory Usage (tokens)', tokens=True, total_type='memory usage')
fig.show()

In [186]:
fig.write_image('figures/frontier_total_token_memory_usage.pdf', width=1500, height=400)

In [139]:
fig = plot_reasoning_length_one_fig(df_reasoning, reasoning_type='Total Reasoning', title='Total Reasoning Length', include_context=False)
fig.show()